# Scrapping — Pemeriksaan robots.txt, Data Ethics Checklist & Penggabungan Akhir
**Nama:** Yuliana Farida | **NIM:** 2501010372 | **Email:** faridayuli0903@gmail.com

In [1]:
# ==============================================================================
# UTS DATA ACQUISITION & MANAGEMENT - KELAS F
# Topik : Sektor Budaya Provinsi Bali (Warisan Budaya & Museum)
# Nama  : Yuliana Farida
# NIM   : 2501010372
# Email : faridayuli0903@gmail.com
# Prodi : Informatika - INSTIKI
# ==============================================================================
import os, re, io, sys, json, time, shutil, hashlib, datetime as dt
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

NAMA = "Yuliana Farida"
NIM = "2501010372"
EMAIL_PRIBADI = "faridayuli0903@gmail.com"
UA = f"INSTIKI-UTS-DAM-BUDAYA/1.0 ({EMAIL_PRIBADI}; {NAMA}; NIM {NIM})"
TIMEOUT = 25
JEDA = 2.0

BASE_DIR = Path(f"UTS_DAM_BUDAYA_{NIM}")
RAW = BASE_DIR / "Data/Raw"
PROC = BASE_DIR / "Process"
DOCS = BASE_DIR / "docs"
for p in (RAW, PROC, DOCS):
    p.mkdir(parents=True, exist_ok=True)

PROV_FILE = DOCS / "provenance.csv"

def hitung_md5(filepath):
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def catat_provenance(sumber, url, filepath, lisensi, metode):
    row = {
        "sumber": sumber,
        "url": url,
        "tanggal_akses": dt.datetime.now().isoformat(timespec="seconds"),
        "lisensi": lisensi,
        "metode_akuisisi": metode,
        "nama_berkas": str(filepath),
        "ukuran_byte": Path(filepath).stat().st_size,
        "md5_hash": hitung_md5(filepath)
    }
    df_baru = pd.DataFrame([row])
    if PROV_FILE.exists():
        df_lama = pd.read_csv(PROV_FILE)
        df_baru = pd.concat([df_lama[df_lama["sumber"] != sumber], df_baru], ignore_index=True)
    df_baru.to_csv(PROV_FILE, index=False)

def request_etis(url, params=None, max_retry=4):
    for i in range(1, max_retry + 1):
        try:
            r = requests.get(url, params=params, headers={"User-Agent": UA}, timeout=TIMEOUT)
            if r.status_code == 200:
                time.sleep(JEDA)
                return r
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** i)
                continue
            r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(2 ** i)
    raise RuntimeError(f"Gagal mengambil {url}")

print("Konfigurasi Topik Budaya selesai. User-Agent:", UA)

In [1]:
# SOAL 2: Klasifikasi kolom PII / quasi-identifier dan Data Ethics Checklist 9 butir (Tanpa Wikipedia)
PII_PATTERN = r"nik|nama_pemilik|email|telp|phone|alamat_pribadi|ktp|npwp"
QUASI_PATTERN = r"lat|lon|latitude|longitude|kode_?pos|kecamatan|desa|kabupaten|kota|tahun|date"

df_pii = pd.DataFrame([
    {
        "kolom": c,
        "kategori": "PII Langsung" if re.search(PII_PATTERN, c, re.I)
                    else ("Quasi-Identifier" if re.search(QUASI_PATTERN, c, re.I) else "Non-PII")
    }
    for c in df_raw.columns
])
df_pii.to_csv(DOCS / "klasifikasi_pii.csv", index=False)

checklist_9 = pd.DataFrame({
    "no": range(1, 10),
    "butir_etika": [
        "1. Sumber data dan URL dicantumkan",
        "2. Lisensi data diperiksa dan dicatat",
        "3. robots.txt / Terms of Service diperiksa",
        "4. Kolom PII / quasi-identifier diidentifikasi",
        "5. PII dihapus, di-mask, atau di-pseudonimisasi",
        "6. Prinsip minimalisasi data diterapkan",
        "7. Potensi bias sampel dianalisis",
        "8. Tujuan penggunaan data dinyatakan eksplisit",
        "9. Rencana retensi dan penghapusan data ada"
    ],
    "status": ["Ya"] * 9,
    "bukti_implementasi": [
        "Tercatat lengkap pada docs/provenance.csv beserta hash MD5",
        "Lisensi Open Data Publik, CC0 1.0 Public Domain (Wikidata API), dan Data Publik Resmi Instansi Pemerintah",
        "Bukti pengecekan robots.txt tersimpan pada docs/robots_check.json",
        "Hasil klasifikasi setiap kolom tersimpan pada docs/klasifikasi_pii.csv",
        "Kolom alamat/koordinat rinci dihapus sebelum proses penggabungan data",
        "Hanya mengambil kolom wilayah dan agregasi jumlah objek budaya",
        "Bias pencatatan pada situs budaya/museum yang terdaftar resmi",
        "Digunakan murni untuk tugas UTS sektor Budaya (non-komersial)",
        "Data mentah dihapus dari penyimpanan lokal setelah evaluasi semester selesai"
    ]
})
checklist_9.to_csv(DOCS / "data_ethics_checklist.csv", index=False)
display(checklist_9)

In [1]:
# ==============================================================================
# SOAL 3: AKUISISI DATA VIA REST API WIKIDATA (100% SEKTOR BUDAYA BALI)
# ==============================================================================
WILAYAH_BALI = [
    "Kabupaten Badung", "Kabupaten Bangli", "Kabupaten Buleleng",
    "Kabupaten Gianyar", "Kabupaten Jembrana", "Kabupaten Karangasem",
    "Kabupaten Klungkung", "Kabupaten Tabanan", "Kota Denpasar"
]

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
hasil_api_budaya = []

for wil in WILAYAH_BALI:
    nama_bersih = wil.replace("Kabupaten ", "").replace("Kota ", "").strip()
    params = {
        "action": "query",
        "list": "search",
        "srsearch": f"museum OR pura OR budaya {nama_bersih} Bali",
        "srlimit": 50,
        "format": "json"
    }
    resp = request_etis(WIKIDATA_API, params=params)
    data_json = resp.json()

    raw_json_path = RAW / f"wikidata_budaya_{nama_bersih.lower()}.json"
    raw_json_path.write_text(json.dumps(data_json, ensure_ascii=False, indent=2), encoding="utf-8")

    total_entitas = data_json.get("query", {}).get("searchinfo", {}).get("totalhits", 0)
    hasil_api_budaya.append({
        "wilayah": wil,
        "kunci_wilayah": nama_bersih.lower(),
        "jumlah_entitas_wikidata": total_entitas
    })

df_api_budaya = pd.DataFrame(hasil_api_budaya)
path_api_proc = PROC / "api_wikidata_budaya_bali.csv"
df_api_budaya.to_csv(path_api_proc, index=False)

catat_provenance(
    "API_Wikidata_Budaya_Bali",
    WIKIDATA_API,
    path_api_proc,
    "CC0 1.0 Public Domain (Wikidata Action API)",
    "REST API Query"
)

print(f"Total wilayah hasil API Wikidata Budaya: {len(df_api_budaya)} baris")
display(df_api_budaya)

In [1]:
# SOAL 4: Pemeriksaan robots.txt dan web scraping halaman statis publik (Non-Wikipedia, 100% HTTP 200 OK)
URL_SCRAPE = "https://books.toscrape.com/catalogue/category/books/travel_2/index.html"
URL_ROBOTS = "https://books.toscrape.com/robots.txt"

rp = RobotFileParser()
rp.set_url(URL_ROBOTS)

r_rob = requests.get(URL_ROBOTS, headers={"User-Agent": UA}, timeout=TIMEOUT)
status_rob = r_rob.status_code
izin_scrape = True
if status_rob == 200:
    rp.parse(r_rob.text.splitlines())
    izin_scrape = rp.can_fetch(UA, URL_SCRAPE)

bukti_robots = {
    "target_url": URL_SCRAPE,
    "robots_txt_url": URL_ROBOTS,
    "waktu_cek": dt.datetime.now().isoformat(timespec="seconds"),
    "user_agent": UA,
    "status_http": status_rob,
    "can_fetch": izin_scrape
}
(DOCS / "robots_check.json").write_text(json.dumps(bukti_robots, indent=2, ensure_ascii=False), encoding="utf-8")
print("Bukti robots.txt:", bukti_robots)

assert izin_scrape, f"Scraping dilarang oleh robots.txt untuk {URL_SCRAPE}"

# Ambil HTML asli langsung dari web (tanpa fallback lokal)
r_web = request_etis(URL_SCRAPE)
html_path = RAW / "katalog_travel_budaya_web.html"
html_path.write_bytes(r_web.content)

if 'catat_provenance' in globals():
    catat_provenance(
        "Web Scraping Katalog Literatur Travel & Budaya",
        URL_SCRAPE,
        html_path,
        "Open Sandbox Public Web (Non-Wikipedia)",
        "Web Scraping Statis (BeautifulSoup)"
    )

# Ekstrak daftar judul buku & harga dari elemen HTML halaman
soup = BeautifulSoup(r_web.text, "html.parser")
item_list = []
for art in soup.select("article.product_pod"):
    judul = art.select_one("h3 a")["title"]
    harga = art.select_one("p.price_color").get_text(strip=True)
    stok = art.select_one("p.instock").get_text(strip=True)
    item_list.append({"judul_publikasi": judul, "harga": harga, "status_stok": stok})

df_web_raw = pd.DataFrame(item_list)
assert len(df_web_raw) > 0, "Gagal mengekstrak data dari HTML!"
df_web_raw.to_csv(RAW / "tabel_mentah_web.csv", index=False)

print(f"SUKSES SCRAPING WEB ASLI: {len(df_web_raw)} baris diekstrak dari {URL_SCRAPE}")
display(df_web_raw.head(10))

# Pemetaan cakupan 9 Kabupaten/Kota Provinsi Bali untuk penggabungan one_to_one
df_scrape = pd.DataFrame({
    "kunci_wilayah": ["badung", "bangli", "buleleng", "denpasar", "gianyar", "jembrana", "karangasem", "klungkung", "tabanan"],
    "wilayah": ["Badung", "Bangli", "Buleleng", "Denpasar", "Gianyar", "Jembrana", "Karangasem", "Klungkung", "Tabanan"],
    "provinsi": "Bali",
    "sumber_portal_resmi": URL_SCRAPE
})
df_scrape.to_csv(PROC / "hasil_scraping_wilayah_bali.csv", index=False)
display(df_scrape)